# Stay At It — Training with your uploaded datasets
Uses `dataset/task_b_sentiment.csv`, merged from `LPC_Statement.csv` and `UT5-Statement.xlsx`.
**2,224 synthetic statements**, with inferred labels that need human verification.
Run cells from top to bottom. Training scores are produced only when executed.


# XLM-RoBERTa fine-tuning (Task A or B)
Run on **Google Colab GPU** if possible. This notebook trains the model specified in your paper. Set `TASK` to `A` or `B`. Run the baseline notebooks first.

**Important:** Task A requires an independently labeled 2,000-listing dataset. Task B currently contains synthetic statements. Results are not known until you execute training.

In [ ]:
# In Colab, uncomment this line:
# !pip -q install transformers accelerate sentencepiece torch pandas scikit-learn matplotlib
import os, random, json, re, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
TASK='B'  # 'A' = listings, 'B' = sentiment
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PATH=ROOT/'dataset'/('task_a_listings.csv' if TASK=='A' else 'task_b_sentiment.csv')
MAX_LEN=256 if TASK=='A' else 128
MODEL_NAME='FacebookAI/xlm-roberta-base'


In [ ]:
df=pd.read_csv(PATH).dropna(subset=['text','label'])
def common_clean(t):
    t=unicodedata.normalize('NFKC',str(t))
    t=re.sub(r'https?://\S+|www\.\S+',' URL ',t)
    t=re.sub(r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}',' EMAIL ',t)
    t=re.sub(r'(?:\+63|0)9\d{9}\b',' PHONE ',t)
    t=re.sub(r'₱\s*[\d,]+(?:\.\d+)?',' PRICE ',t)
    return re.sub(r'\s+',' ',t).strip()
df['text']=df.text.map(common_clean)
df=df.drop_duplicates(subset='text').reset_index(drop=True)
labels=sorted(df.label.unique());to_id={s:i for i,s in enumerate(labels)}
df['target']=df.label.map(to_id)
trainval,test=train_test_split(df,test_size=.15,stratify=df.target,random_state=42)
train,val=train_test_split(trainval,test_size=.15/.85,stratify=trainval.target,random_state=42)
print('Split:',len(train),len(val),len(test),'Labels:',labels)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
class TextData(torch.utils.data.Dataset):
    def __init__(self,frame):
        self.enc=tokenizer(frame.text.tolist(),truncation=True,max_length=MAX_LEN)
        self.y=frame.target.tolist()
    def __len__(self):return len(self.y)
    def __getitem__(self,i):return {**{k:v[i] for k,v in self.enc.items()},'labels':self.y[i]}
train_ds,val_ds,test_ds=map(TextData,[train,val,test])
counts=train.target.value_counts().reindex(range(len(labels)),fill_value=1)
weights=torch.tensor(len(train)/(len(labels)*counts.values),dtype=torch.float32)


In [ ]:
def metrics(eval_pred):
    logits,truth=eval_pred
    pred=np.argmax(logits,axis=-1)
    p,r,f,_=precision_recall_fscore_support(truth,pred,average='macro',zero_division=0)
    return {'accuracy':accuracy_score(truth,pred),'macro_precision':p,'macro_recall':r,'macro_f1':f}
class WeightedTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        labels=inputs.pop('labels')
        outputs=model(**inputs)
        loss=torch.nn.functional.cross_entropy(outputs.logits,labels,weight=weights.to(outputs.logits.device))
        return (loss,outputs) if return_outputs else loss


In [ ]:
# Train 3 seeds; each model selects its best epoch using validation macro F1.
# Note: transformers versions differ; this notebook targets current Hugging Face APIs.
all_runs=[]
for seed in [13,42,99]:
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=len(labels),id2label=dict(enumerate(labels)),label2id=to_id)
    out=ROOT/'model'/f'xlmr_task_{TASK.lower()}_seed_{seed}'
    args=TrainingArguments(output_dir=str(out),learning_rate=2e-5,per_device_train_batch_size=16,per_device_eval_batch_size=16,num_train_epochs=4,weight_decay=.01,warmup_ratio=.1,eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=1,report_to='none',seed=seed,fp16=torch.cuda.is_available())
    trainer=WeightedTrainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=metrics)
    trainer.train()
    # Each seed's untouched test is evaluated once after selecting its epoch.
    outcome=trainer.predict(test_ds)
    scores=metrics((outcome.predictions,outcome.label_ids))
    all_runs.append({'seed':seed,**scores})
    trainer.save_model(str(out));tokenizer.save_pretrained(str(out))
    pd.DataFrame(all_runs).to_csv(ROOT/'results'/f'xlmr_task_{TASK.lower()}_runs.csv',index=False)
print(pd.DataFrame(all_runs))
print('Mean and SD:');print(pd.DataFrame(all_runs).drop(columns='seed').agg(['mean','std']))


### Paper reporting checklist
- Report accuracy, macro precision/recall/F1 for both baselines and transformer.
- Compute per-class F1 and confusion matrices from held-out predictions.
- Use **1,000 paired bootstrap samples** on identical test examples to estimate the XLM-R vs best baseline macro-F1 difference (95% CI).
- Review misclassified Taglish texts for negation, short messages, code-switching and misspellings.
- Do not report a 5-point improvement unless actual results and confidence interval support it.